# BigQuery Extraction Explorer

Interactive analysis of `bq-extraction` results.  
Loads one or more extraction-run folders, normalises their JSON artifacts, and produces inventory, workload, cost, and access visualisations.

**Sections**
1. Setup & data loading
2. Run overview
3. Dataset inventory & governance labels
4. Table inventory & storage
5. Workload mix by query source
6. Hot tables (access frequency & breadth)
7. Top users by volume & cost
8. Repeated-query patterns
9. Cross-project references
10. Atlas / Alma bridge notes

## 1. Setup & Data Loading

Point `RESULTS_DIR` at the directory containing your extraction run folders.  
Default: `output/` (the extractor's default output path).  
Override if results live elsewhere, e.g. `../velum-extraction-results`.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from bq_extraction.loader import load_runs, filter_probe_jobs

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)

In [ ]:
RESULTS_DIR = Path("../output")

runs = load_runs(RESULTS_DIR)
print(f"Loaded {len(runs.run_summary)} run(s) from {RESULTS_DIR.resolve()}")
runs.run_summary

## 2. Run Overview

Quick health check: which runs are complete (all 10 artifact files) vs partial (datasets-only, etc.).

In [ ]:
summary = runs.run_summary.copy()
summary["artifacts_str"] = summary["artifacts"].apply(lambda xs: ", ".join(xs))

fig = px.bar(
    summary,
    x="run_id",
    y="artifact_count",
    color="is_complete",
    text="artifact_count",
    hover_data=["project_id", "artifacts_str"],
    title="Artifacts per extraction run",
    labels={"artifact_count": "Artifact count", "run_id": "Run"},
    color_discrete_map={True: "#2ecc71", False: "#e74c3c"},
)
fig.update_layout(xaxis_tickangle=-30, showlegend=False)
fig.show()

## 3. Dataset Inventory & Governance Labels

Overview of discovered datasets across all runs, grouped by project.  
Datasets with `maturity=contract` and a `producer` label already have some governance metadata attached.

In [ ]:
if runs.datasets is not None:
    ds = runs.datasets.copy()
    print(f"Total dataset records: {len(ds)} across {ds['project_id'].nunique()} project(s)")
    display(ds.groupby("project_id").agg(
        datasets=("dataset_id", "nunique"),
        with_maturity_label=("label_maturity", lambda s: (s != "").sum()),
        with_producer_label=("label_producer", lambda s: (s != "").sum()),
    ).reset_index())
else:
    print("No datasets artifact found.")

In [ ]:
if runs.datasets is not None:
    ds = runs.datasets.copy()
    ds["has_contract"] = ds["label_maturity"] == "contract"
    label_counts = ds.groupby(["project_id", "has_contract"]).size().reset_index(name="count")

    fig = px.bar(
        label_counts,
        x="project_id",
        y="count",
        color="has_contract",
        barmode="stack",
        title="Dataset count by project (contract-labelled vs unlabelled)",
        labels={"count": "Datasets", "project_id": "Project", "has_contract": "Has contract label"},
        color_discrete_map={True: "#2ecc71", False: "#95a5a6"},
    )
    fig.update_layout(xaxis_tickangle=-20)
    fig.show()

## 4. Table Inventory & Storage

Breakdown of tables/views per project and dataset, ranked by storage size.

In [ ]:
if runs.tables is not None:
    tbl = runs.tables.copy()
    print(f"Total table/view records: {len(tbl)} across {tbl['project_id'].nunique()} project(s)")

    type_counts = tbl.groupby(["project_id", "table_type"]).size().reset_index(name="count")
    fig = px.bar(
        type_counts,
        x="project_id",
        y="count",
        color="table_type",
        barmode="stack",
        title="Tables & views per project",
        labels={"count": "Objects", "project_id": "Project"},
    )
    fig.update_layout(xaxis_tickangle=-20)
    fig.show()
else:
    print("No tables artifact found.")

In [ ]:
if runs.tables is not None:
    tbl = runs.tables.copy()
    tbl["gb"] = tbl["num_bytes"].fillna(0) / 1e9

    top_tables = (
        tbl.nlargest(30, "gb")
        .assign(fqn=lambda d: d["project_id"] + "." + d["dataset_id"] + "." + d["table_id"])
    )

    fig = px.bar(
        top_tables,
        x="fqn",
        y="gb",
        color="project_id",
        hover_data=["num_rows", "column_count", "table_type"],
        title="Top 30 tables by storage (GB)",
        labels={"gb": "Size (GB)", "fqn": "Table"},
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

## 5. Workload Mix by Query Source

How is query volume distributed across `service_account`, `ad_hoc`, `scheduled`, and `batch` sources?

In [ ]:
if runs.query_sources is not None:
    qs = runs.query_sources.copy()

    fig = px.bar(
        qs,
        x="project_id",
        y="query_count",
        color="query_source",
        barmode="group",
        hover_data=["total_gb_processed", "total_slot_hours"],
        title="Query count by source type and project",
        labels={"query_count": "Queries", "project_id": "Project"},
    )
    fig.show()
else:
    print("No query_sources artifact found.")

In [ ]:
if runs.query_sources is not None:
    qs = runs.query_sources.copy()

    fig = px.bar(
        qs,
        x="project_id",
        y="total_slot_hours",
        color="query_source",
        barmode="group",
        title="Slot-hours by source type and project",
        labels={"total_slot_hours": "Slot-hours", "project_id": "Project"},
    )
    fig.show()

## 6. Hot Tables (Access Frequency & Breadth)

Which tables drive the most query volume? How many distinct users touch them?

In [ ]:
if runs.table_access is not None:
    ta = runs.table_access.copy()
    ta["fqn"] = ta["project_id"] + "." + ta["dataset_id"] + "." + ta["table_id"]

    ta = ta[~ta["dataset_id"].str.startswith("region-")].copy()

    top_access = ta.nlargest(30, "query_count")

    fig = px.bar(
        top_access,
        x="fqn",
        y="query_count",
        color="distinct_users",
        hover_data=["first_accessed", "last_accessed", "project_id"],
        title="Top 30 most-queried tables (colour = distinct users)",
        labels={"query_count": "Query count", "fqn": "Table", "distinct_users": "Distinct users"},
        color_continuous_scale="Viridis",
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()
else:
    print("No table_access artifact found.")

In [ ]:
if runs.table_access is not None:
    ta = runs.table_access.copy()
    ta = ta[~ta["dataset_id"].str.startswith("region-")].copy()

    per_dataset = (
        ta.groupby(["project_id", "dataset_id"])
        .agg(total_queries=("query_count", "sum"), tables=("table_id", "nunique"))
        .reset_index()
        .nlargest(20, "total_queries")
    )

    fig = px.bar(
        per_dataset,
        x="dataset_id",
        y="total_queries",
        color="project_id",
        hover_data=["tables"],
        title="Top 20 datasets by total query volume",
        labels={"total_queries": "Total queries", "dataset_id": "Dataset"},
    )
    fig.update_layout(xaxis_tickangle=-30)
    fig.show()

## 7. Top Users by Volume & Cost

Which service accounts and humans generate the most load?

In [ ]:
if runs.user_stats is not None:
    us = runs.user_stats.copy()
    us["user_type"] = us["user_email"].apply(
        lambda e: "service_account" if "gserviceaccount.com" in str(e) or "@appspot" in str(e) else "human"
    )
    us["short_email"] = us["user_email"].str.split("@").str[0]

    top_users = us.nlargest(20, "query_count")

    fig = px.bar(
        top_users,
        x="short_email",
        y="query_count",
        color="user_type",
        hover_data=["project_id", "total_gb_processed", "total_slot_hours", "dml_count"],
        title="Top 20 users by query count",
        labels={"query_count": "Queries", "short_email": "User"},
        color_discrete_map={"service_account": "#3498db", "human": "#e67e22"},
    )
    fig.update_layout(xaxis_tickangle=-30)
    fig.show()
else:
    print("No user_stats artifact found.")

In [ ]:
if runs.user_stats is not None:
    us = runs.user_stats.copy()
    us["user_type"] = us["user_email"].apply(
        lambda e: "service_account" if "gserviceaccount.com" in str(e) or "@appspot" in str(e) else "human"
    )
    us["short_email"] = us["user_email"].str.split("@").str[0]

    top_cost = us.nlargest(20, "total_slot_hours")

    fig = px.scatter(
        top_cost,
        x="total_gb_processed",
        y="total_slot_hours",
        size="query_count",
        color="user_type",
        hover_name="short_email",
        hover_data=["project_id"],
        title="Users: GB processed vs slot-hours (size = query count)",
        labels={"total_gb_processed": "Total GB processed", "total_slot_hours": "Total slot-hours"},
        color_discrete_map={"service_account": "#3498db", "human": "#e67e22"},
    )
    fig.show()

## 8. Repeated-Query Patterns

Top query hashes by execution count with avg bytes and slot cost. These are the best candidates for caching, materialisation, or contract extraction.

In [ ]:
if runs.frequent_queries is not None:
    fq = runs.frequent_queries.copy()
    fq["avg_gb"] = fq["avg_bytes"].fillna(0) / 1e9
    fq["total_est_gb"] = fq["avg_gb"] * fq["execution_count"]

    top_fq = fq.nlargest(25, "execution_count")

    fig = px.scatter(
        top_fq,
        x="execution_count",
        y="avg_slot_ms",
        size="avg_gb",
        color="project_id",
        hover_data=["query_hash", "total_est_gb"],
        title="Top 25 frequent queries: execution count vs avg slot-ms (size = avg GB)",
        labels={"execution_count": "Executions", "avg_slot_ms": "Avg slot-ms", "avg_gb": "Avg GB"},
    )
    fig.show()
else:
    print("No frequent_queries artifact found.")

In [ ]:
if runs.frequent_queries is not None:
    fq = runs.frequent_queries.copy()
    fq["avg_gb"] = fq["avg_bytes"].fillna(0) / 1e9
    fq["total_est_gb"] = fq["avg_gb"] * fq["execution_count"]

    cumulative = fq.sort_values("execution_count", ascending=False).reset_index(drop=True)
    cumulative["cumulative_pct"] = (
        cumulative["execution_count"].cumsum() / cumulative["execution_count"].sum() * 100
    )
    cumulative["rank"] = range(1, len(cumulative) + 1)

    fig = px.line(
        cumulative,
        x="rank",
        y="cumulative_pct",
        title="Pareto: cumulative share of total executions by query-hash rank",
        labels={"rank": "Query hash rank", "cumulative_pct": "Cumulative % of executions"},
    )
    fig.add_hline(y=80, line_dash="dash", annotation_text="80%")
    fig.show()

In [ ]:
if runs.frequent_queries is not None:
    fq = runs.frequent_queries.copy()
    fq["avg_gb"] = fq["avg_bytes"].fillna(0) / 1e9
    fq["total_est_gb"] = fq["avg_gb"] * fq["execution_count"]
    fq["query_preview"] = fq["sample_query"].str[:200]
    fq["user_list"] = fq["users"].apply(lambda xs: ", ".join(xs) if isinstance(xs, list) else str(xs))

    display(
        fq.nlargest(15, "execution_count")[
            ["project_id", "execution_count", "avg_gb", "total_est_gb", "avg_slot_ms", "user_list", "query_preview"]
        ]
    )

## 9. Cross-Project References

Queries in one project that reference tables in a different project reveal cross-project dependencies.  
Probe/self-generated queries (INFORMATION_SCHEMA) are excluded.

In [ ]:
if runs.query_logs is not None:
    ql = filter_probe_jobs(runs.query_logs.copy())
    ql_refs = ql[ql["referenced_tables"].apply(bool)].copy()

    exploded = ql_refs.explode("referenced_tables").reset_index(drop=True)
    exploded["ref_project"] = exploded["referenced_tables"].apply(
        lambda r: r.get("project_id", "") if isinstance(r, dict) else ""
    )
    exploded["ref_dataset"] = exploded["referenced_tables"].apply(
        lambda r: r.get("dataset_id", "") if isinstance(r, dict) else ""
    )
    exploded["ref_table"] = exploded["referenced_tables"].apply(
        lambda r: r.get("table_id", "") if isinstance(r, dict) else ""
    )

    xdf = exploded[(exploded["ref_project"] != "") & (exploded["ref_project"] != exploded["project_id"])]

    if not xdf.empty:
        print(f"Found {len(xdf)} cross-project table references")

        edge_counts = (
            xdf.groupby(["project_id", "ref_project"])
            .agg(references=("ref_table", "count"), distinct_tables=("ref_table", "nunique"))
            .reset_index()
        )
        display(edge_counts)

        top_external = (
            xdf.groupby(["ref_project", "ref_dataset", "ref_table"])
            .size()
            .reset_index(name="ref_count")
            .nlargest(20, "ref_count")
        )
        display(top_external)
    else:
        print("No cross-project references found in query logs.")
else:
    print("No query_logs artifact found.")

In [ ]:
if runs.query_logs is not None:
    ql = filter_probe_jobs(runs.query_logs.copy())
    ql = ql.dropna(subset=["creation_time"])

    if not ql.empty:
        ql["date"] = ql["creation_time"].dt.date
        daily = ql.groupby(["project_id", "date"]).size().reset_index(name="queries")

        fig = px.line(
            daily,
            x="date",
            y="queries",
            color="project_id",
            title="Daily query volume by project (probes excluded)",
            labels={"date": "Date", "queries": "Queries"},
        )
        fig.show()

## 10. Atlas / Alma Bridge Notes

The normalised DataFrames produced by the loader are shaped to map cleanly into downstream systems:

| Notebook frame | Atlas target | Alma / Observatory target |
|---|---|---|
| `datasets` + `tables` | `assets` table in `atlas.db` (kind=table/view, source=bigquery) | Asset registry via `CreateSourceAdapter` + `IntrospectSourceAdapterSchema` |
| `frequent_queries` | `queries` table (`fingerprint` = `query_hash`, `execution_count`, `tables` from sample SQL) | `QuerySignature` proto (relations, joins, predicates extracted from `sample_query`) |
| `user_stats` | `consumers` table (kind=user or service, one row per email) | `QueryEvent.source_name` mapped to consumer identity |
| `table_access` | `edges` table (upstream=table asset, downstream=consumer, kind=reads) + `consumer_assets` | `TrafficStats.by_table` / lineage edge storage (`data_edge`) |
| `query_logs` | raw traffic for `alma-algebrakit` fingerprinting and lineage assembly | `QueryEvent` messages for `Ingest` RPC / observation ingestion |
| `ddls` | `schema_snapshots` (columns parsed from DDL text) | Source adapter schema snapshots |

### How to hydrate Atlas

1. Map each `(project_id, dataset_id, table_id)` from `tables` into an Atlas `assets` row with `id = project.dataset.table`, `source = 'bigquery'`, `kind = TABLE or VIEW`.
2. Map each `frequent_queries` row into `queries` using `query_hash` as `fingerprint` and the JSON list of referenced tables as `tables`.
3. Map each `user_stats` email into `consumers` with `kind = 'service'` or `'user'`.
4. Map `table_access` rows into `edges` (consumer -> asset, kind=reads) and `consumer_assets`.
5. Run `alma-atlas serve` to expose the populated graph via MCP.

### How to adapt toward Alma / Observatory

1. Transform `query_logs` rows into `QueryEvent` protobuf messages (`sql`, `duration_ms`, `fingerprint_hash`, `target_id`, `backend_system='bigquery'`).
2. Use `query-analyzer` to parse `sample_query` from `frequent_queries` into `QuerySignature` (relations, join_edges, predicates, projections).
3. Aggregate `query_sources` and `user_stats` into `TrafficStats` for the Observatory dashboard.
4. Use `table_access` to seed `data_edge` records in the Observatory edge storage.
5. Use dataset `labels` to bootstrap contract status: datasets with `maturity=contract` already have governance metadata.

## 11. Offline Lineage Graph

Build a neutral lineage graph from the extracted artifacts, then derive an Atlas-ready export payload from it.  
This section focuses on three things:
- who reads what
- what writes or materializes into what
- where view and cross-project dependencies exist

In [ ]:
from bq_extraction.atlas_export import build_atlas_export
from bq_extraction.lineage import build_lineage_graph

lineage_graph = build_lineage_graph(runs)
atlas_export = build_atlas_export(lineage_graph, source_id="bq-extraction-offline")

lineage_nodes_df, lineage_edges_df, lineage_queries_df, lineage_issues_df = lineage_graph.to_dataframes()

print(
    f"Lineage graph: {len(lineage_graph.nodes)} node(s), {len(lineage_graph.edges)} edge(s), "
    f"{len(lineage_graph.queries)} query pattern(s), {len(lineage_graph.issues)} issue(s)"
)
print(
    f"Atlas export: {len(atlas_export.assets)} asset(s), {len(atlas_export.edges)} edge(s), "
    f"{len(atlas_export.queries)} query row(s), {len(atlas_export.consumers)} consumer(s)"
)

In [ ]:
edge_summary = (
    lineage_edges_df.groupby("edge_type")
    .size()
    .reset_index(name="edge_count")
    .sort_values("edge_count", ascending=False)
)
display(edge_summary)

if not lineage_issues_df.empty:
    display(lineage_issues_df.head(20))

In [ ]:
reads_df = lineage_edges_df[lineage_edges_df["edge_type"] == "reads"].copy()

if not reads_df.empty:
    reads_df["consumer"] = reads_df["target"].str.removeprefix("consumer:")
    reads_df["source_project"] = reads_df["source"].str.split(".").str[0]
    reads_df["query_count"] = reads_df["metadata"].apply(lambda m: m.get("query_count", 0.0))
    reads_df["cross_project"] = reads_df["metadata"].apply(lambda m: m.get("cross_project", False))

    top_consumers = (
        reads_df.groupby("consumer")
        .agg(query_count=("query_count", "sum"), tables=("source", "nunique"))
        .reset_index()
        .nlargest(20, "query_count")
    )

    fig = px.bar(
        top_consumers,
        x="consumer",
        y="query_count",
        color="tables",
        title="Top consumers by read-lineage query count",
        labels={"query_count": "Queries", "consumer": "Consumer", "tables": "Distinct tables"},
        color_continuous_scale="Viridis",
    )
    fig.update_layout(xaxis_tickangle=-35)
    fig.show()
else:
    print("No read lineage edges available.")

In [ ]:
if not reads_df.empty:
    cross_project_reads = reads_df[reads_df["cross_project"]].copy()

    if not cross_project_reads.empty:
        cross_project_reads["observed_project"] = cross_project_reads["metadata"].apply(
            lambda m: ", ".join(m.get("observed_projects", [])) if m.get("observed_projects") else ""
        )
        cross_project_summary = (
            cross_project_reads.groupby(["observed_project", "source_project"])
            .agg(query_count=("query_count", "sum"), consumers=("consumer", "nunique"), tables=("source", "nunique"))
            .reset_index()
            .sort_values("query_count", ascending=False)
        )
        display(cross_project_summary)
    else:
        print("No cross-project read edges found.")

writes_df = lineage_edges_df[lineage_edges_df["edge_type"] == "writes"].copy()
if not writes_df.empty:
    writes_df["target_project"] = writes_df["target"].str.split(".").str[0]
    writes_df["query_count"] = writes_df["metadata"].apply(lambda m: m.get("query_count", 0.0))
    top_write_targets = (
        writes_df.groupby("target")
        .agg(query_count=("query_count", "sum"), upstream_tables=("source", "nunique"))
        .reset_index()
        .nlargest(20, "query_count")
    )

    fig = px.bar(
        top_write_targets,
        x="target",
        y="query_count",
        color="upstream_tables",
        title="Top write targets from SQL heuristics",
        labels={"target": "Target table", "query_count": "Queries", "upstream_tables": "Upstream tables"},
        color_continuous_scale="Magma",
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()
else:
    print("No write-lineage edges found.")

In [ ]:
view_df = lineage_edges_df[lineage_edges_df["edge_type"] == "view_depends_on"].copy()

if not view_df.empty:
    view_df["upstream_count"] = 1
    view_summary = (
        view_df.groupby("target")
        .agg(upstream_tables=("source", "nunique"))
        .reset_index()
        .nlargest(20, "upstream_tables")
    )
    display(view_summary)
else:
    print("No view dependency edges found.")

centrality_df = lineage_edges_df.copy()
if not centrality_df.empty:
    out_degree = centrality_df.groupby("source").size().reset_index(name="out_degree")
    in_degree = centrality_df.groupby("target").size().reset_index(name="in_degree")
    degree_df = out_degree.merge(in_degree, left_on="source", right_on="target", how="outer")
    degree_df["node"] = degree_df["source"].fillna(degree_df["target"])
    degree_df["out_degree"] = degree_df["out_degree"].fillna(0)
    degree_df["in_degree"] = degree_df["in_degree"].fillna(0)
    degree_df = degree_df[["node", "out_degree", "in_degree"]]

    display(degree_df.sort_values(["in_degree", "out_degree"], ascending=False).head(20))

## 12. Atlas Export Preview

This is the Atlas-shaped payload derived from the neutral lineage graph.  
Use it to sanity-check counts and sample rows before writing JSON or SQLite artifacts.

In [ ]:
atlas_assets_df = pd.DataFrame(atlas_export.assets)
atlas_edges_df = pd.DataFrame(atlas_export.edges)
atlas_queries_df = pd.DataFrame(atlas_export.queries)
atlas_consumers_df = pd.DataFrame(atlas_export.consumers)
atlas_schema_df = pd.DataFrame(atlas_export.schema_snapshots)

print(f"assets={len(atlas_assets_df)} edges={len(atlas_edges_df)} queries={len(atlas_queries_df)} consumers={len(atlas_consumers_df)} schema_snapshots={len(atlas_schema_df)}")

display(atlas_assets_df.head(10))
display(atlas_edges_df.head(10))
display(atlas_queries_df.head(10))

# Optional: write artifacts from the notebook
# lineage_graph.write_json("../output/lineage-notebook/neutral")
# atlas_export.write_json("../output/lineage-notebook/atlas")
# atlas_export.write_sqlite("../output/lineage-notebook/atlas.db")

## 13. Notebook Lineage Graph

Render a **bounded subgraph** from the lineage data.  
This is intentionally neighborhood-based: the full graph is too large to draw in a useful way.

Suggested usage:
- start with a single `GRAPH_PROJECT` or `GRAPH_DATASET`
- keep `GRAPH_HOPS = 1`
- include only the edge types you care about
- raise `GRAPH_MAX_NODES` carefully if the neighborhood is too small

In [ ]:
import networkx as nx

from bq_extraction.lineage import extract_lineage_subgraph

lineage_projects = sorted(
    {
        metadata.get("project_id", "")
        for metadata in lineage_nodes_df["metadata"]
        if isinstance(metadata, dict) and metadata.get("project_id")
    }
)
lineage_datasets = sorted(
    {
        node_id
        for node_id, node_type in zip(lineage_nodes_df["id"], lineage_nodes_df["node_type"])
        if node_type == "dataset"
    }
)
lineage_consumers = sorted(
    lineage_nodes_df.loc[lineage_nodes_df["node_type"] == "consumer", "name"].dropna().unique().tolist()
)
lineage_assets = sorted(
    lineage_nodes_df.loc[
        lineage_nodes_df["node_type"].isin(["table", "view", "materialized_view"]), "id"
    ].dropna().unique().tolist()
)

print("Projects:", lineage_projects)
print("Sample datasets:", lineage_datasets[:20])
print("Sample consumers:", lineage_consumers[:15])
print("Sample assets:", lineage_assets[:15])

In [ ]:
GRAPH_PROJECT = lineage_projects[0] if lineage_projects else ""
GRAPH_DATASET = ""          # example: "insightful-parrot.analytics"
GRAPH_CONSUMER = ""         # example: "analyst@example.com"
GRAPH_ASSET = ""            # example: "insightful-parrot.analytics.sessions_metadata"
GRAPH_EDGE_TYPES = ["reads", "writes", "view_depends_on"]
GRAPH_HOPS = 1
GRAPH_MAX_NODES = 80

print({
    "GRAPH_PROJECT": GRAPH_PROJECT,
    "GRAPH_DATASET": GRAPH_DATASET,
    "GRAPH_CONSUMER": GRAPH_CONSUMER,
    "GRAPH_ASSET": GRAPH_ASSET,
    "GRAPH_EDGE_TYPES": GRAPH_EDGE_TYPES,
    "GRAPH_HOPS": GRAPH_HOPS,
    "GRAPH_MAX_NODES": GRAPH_MAX_NODES,
})

In [ ]:
lineage_subgraph = extract_lineage_subgraph(
    lineage_graph,
    project_id=GRAPH_PROJECT or None,
    dataset_id=GRAPH_DATASET or None,
    consumer=GRAPH_CONSUMER or None,
    asset_id=GRAPH_ASSET or None,
    edge_types=GRAPH_EDGE_TYPES,
    hop_depth=GRAPH_HOPS,
    max_nodes=GRAPH_MAX_NODES,
)
subgraph_nodes_df, subgraph_edges_df = lineage_subgraph.to_dataframes()

print(
    f"Subgraph: {lineage_subgraph.graph.number_of_nodes()} node(s), "
    f"{lineage_subgraph.graph.number_of_edges()} edge(s), "
    f"seed(s)={lineage_subgraph.seed_node_ids}"
)
if lineage_subgraph.truncated:
    print(
        f"Warning: subgraph was truncated to {lineage_subgraph.node_limit} nodes; "
        f"{lineage_subgraph.omitted_nodes} node(s) were omitted."
    )
if subgraph_nodes_df.empty:
    print("No nodes matched the current filters. Try narrowing to a real project, dataset, consumer, or asset.")
else:
    display(subgraph_nodes_df[["id", "node_type", "name", "degree", "in_degree", "out_degree"]].head(20))
    display(subgraph_edges_df[["source", "target", "edge_type"]].head(20))

In [ ]:
if lineage_subgraph.graph.number_of_nodes() > 0:
    node_type_colors = {
        "dataset": "#8e44ad",
        "table": "#3498db",
        "view": "#2ecc71",
        "materialized_view": "#16a085",
        "consumer": "#e67e22",
    }
    edge_type_colors = {
        "reads": "#7f8c8d",
        "writes": "#e74c3c",
        "view_depends_on": "#27ae60",
    }

    pos = nx.spring_layout(
        lineage_subgraph.graph,
        seed=42,
        k=max(0.3, 2 / max(2, lineage_subgraph.graph.number_of_nodes() ** 0.5)),
    )

    edge_traces = []
    edge_hover_traces = []
    for edge_type in sorted({data["edge_type"] for _, _, data in lineage_subgraph.graph.edges(data=True)}):
        x_lines = []
        y_lines = []
        x_mid = []
        y_mid = []
        hover_text = []
        for source, target, data in lineage_subgraph.graph.edges(data=True):
            if data["edge_type"] != edge_type:
                continue
            x0, y0 = pos[source]
            x1, y1 = pos[target]
            x_lines += [x0, x1, None]
            y_lines += [y0, y1, None]
            x_mid.append((x0 + x1) / 2)
            y_mid.append((y0 + y1) / 2)
            meta = data.get("metadata", {})
            hover_text.append(
                "<br>".join([
                    f"edge_type={data['edge_type']}",
                    f"source={source}",
                    f"target={target}",
                    f"cross_project={meta.get('cross_project', False)}",
                    f"query_count={meta.get('query_count', '')}",
                    f"provenance={', '.join(meta.get('provenance_sources', [])) if isinstance(meta.get('provenance_sources'), list) else meta.get('provenance_sources', '')}",
                ])
            )

        edge_traces.append(
            go.Scatter(
                x=x_lines,
                y=y_lines,
                mode="lines",
                line={"width": 1.5, "color": edge_type_colors.get(edge_type, "#95a5a6")},
                hoverinfo="skip",
                name=edge_type,
            )
        )
        edge_hover_traces.append(
            go.Scatter(
                x=x_mid,
                y=y_mid,
                mode="markers",
                marker={"size": 8, "color": edge_type_colors.get(edge_type, "#95a5a6"), "opacity": 0.18},
                hovertemplate="%{text}<extra></extra>",
                text=hover_text,
                name=f"{edge_type} details",
                showlegend=False,
            )
        )

    node_x = []
    node_y = []
    node_text = []
    node_color = []
    node_size = []
    node_label = []

    for node_id, data in lineage_subgraph.graph.nodes(data=True):
        x, y = pos[node_id]
        node_x.append(x)
        node_y.append(y)
        metadata = data.get("metadata", {})
        node_text.append(
            "<br>".join([
                f"id={node_id}",
                f"type={data.get('node_type', '')}",
                f"name={data.get('name', '')}",
                f"project_id={metadata.get('project_id', '')}",
                f"dataset_id={metadata.get('dataset_id', '')}",
                f"table_id={metadata.get('table_id', '')}",
                f"consumer_kind={metadata.get('consumer_kind', '')}",
                f"discovered_via={metadata.get('discovered_via', '')}",
            ])
        )
        node_color.append(node_type_colors.get(data.get("node_type", ""), "#34495e"))
        node_size.append(14 + (lineage_subgraph.graph.degree(node_id) * 2))
        node_label.append(data.get("name") or node_id.split(".")[-1])

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers+text",
        text=node_label,
        textposition="top center",
        hovertemplate="%{customdata}<extra></extra>",
        customdata=node_text,
        marker={
            "size": node_size,
            "color": node_color,
            "line": {"width": 1, "color": "#1f2937"},
        },
        name="nodes",
    )

    fig = go.Figure(data=[*edge_traces, *edge_hover_traces, node_trace])
    fig.update_layout(
        title=(
            f"Lineage subgraph ({lineage_subgraph.graph.number_of_nodes()} nodes, "
            f"{lineage_subgraph.graph.number_of_edges()} edges)"
        ),
        showlegend=True,
        hovermode="closest",
        margin={"l": 20, "r": 20, "t": 50, "b": 20},
        xaxis={"showgrid": False, "zeroline": False, "showticklabels": False},
        yaxis={"showgrid": False, "zeroline": False, "showticklabels": False},
        height=750,
    )
    fig.show()
else:
    print("No graph to render for the current filters.")

In [ ]:
if not subgraph_nodes_df.empty:
    if lineage_subgraph.truncated:
        print("Showing fallback tables because the selected neighborhood exceeded the render cap.")

    display(
        subgraph_nodes_df.sort_values(["degree", "node_type", "id"], ascending=[False, True, True]).head(50)
    )
    display(subgraph_edges_df.sort_values(["edge_type", "source", "target"]).head(100))